In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
%run ../00-common/05.data-quality-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
from pyspark.sql import functions as F
from datetime import date
from dateutil.relativedelta import relativedelta

In [0]:
drivers_df = spark.table(bronze_table).filter((F.col("batch_id") == v_batch_id))

In [0]:
drivers_dropped_df = drivers_df.drop(F.col("url"))

In [0]:
drivers_renamed_df = (
    drivers_dropped_df
        .withColumnsRenamed({
            "driverId": "driver_id",
            "dateOfBirth": "date_of_birth"
        })
)

In [0]:
drivers_concatenated_df = (
  drivers_renamed_df
       .withColumn("driver_name", 
                   F.initcap(F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName"))))
       .drop("name")
)

In [0]:
drivers_distinct_df = drivers_concatenated_df.dropDuplicates(["driver_id"])

In [0]:
drivers_final_df = (
    drivers_distinct_df
        .withColumn('nationality', F.initcap(F.col("nationality")))
)

In [0]:
# dynamically calculating the upper bound for dob_range
min_driver_age = 15
max_dob_str    = (date.today() - relativedelta(years=min_driver_age)).strftime("%Y-%m-%d")

dq_checks = [
    # not_null
    {"name": "driver_id_not_null",   "type": "not_null",    "column": "driver_id",   "severity": "critical"},
    {"name": "driver_name_not_null", "type": "not_null",    "column": "driver_name", "severity": "warning"},
    {"name": "nationality_not_null", "type": "not_null",    "column": "nationality", "severity": "warning"},
    # unique
    {"name": "driver_id_unique",     "type": "unique",      "columns": ["driver_id"],"severity": "critical"},
    # min_rows
    {"name": "min_one_row",          "type": "min_rows",    "threshold": 1,          "severity": "warning"},
    # Date of birth check
    # min - earliest F1 era.keeping it as "1900-01-01"
    # max - dynamic, always today minus 18 years (minimum F1 driver age)
    {"name": "dob_range",            "type": "value_range", "column": "date_of_birth",
     "min": "1900-01-01",  "max": max_dob_str,              "severity": "warning"},
]

run_dq_checks(
    df                  = drivers_final_df,
    checks              = dq_checks,
    table_name          = silver_table,
    batch_id            = v_batch_id,
    raise_on_critical   = True
)

In [0]:
apply_delta_constraints(
    table_name  = silver_table,
    constraints = [
        {"name": "chk_driver_id_not_null",   "condition": "driver_id IS NOT NULL"},
        {"name": "chk_driver_name_not_null", "condition": "driver_name IS NOT NULL"},
    ]
)

In [0]:
write_to_silver(
    input_df = drivers_final_df,
    target_table = silver_table,
    merge_condition = "t.driver_id = s.driver_id",
    columns_to_update = [
        "driver_name",
        "date_of_birth",
        "nationality",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)